# NB8 - Live demo: construction-site worker risk scoring (Gradio)
Upload a site photo -> detector + pose + **Depth Pro / GeoCalib** run live -> per-worker PPE / struck-by / fall scores and one combined risk score.
Pick the **distance method** (GeoCalib ground-plane geometry or Depth Pro) with the radio button; both are computed once per image, so switching is instant.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Drive must contain `cv5570_risk/weights/` (detector, thresholds, calibration, `risk_config.json`) and `cv5570_risk/code/risk_pose.py` (written by the pose notebook).
**Run all cells top to bottom (about 3-4 min the first time). The last cell prints a public link.**
Engine code in cells 3-8 is copied unchanged from NB4 (engines + deployment), so scores match the NB4 results.

In [ ]:
# 1. install everything BEFORE importing ultralytics (so YOLO-World's CLIP dependency does not need a runtime restart)
!pip -q install "ultralytics>=8.4.0" gradio ftfy scikit-learn joblib pandas
!pip -q install git+https://github.com/ultralytics/CLIP.git
!pip -q install git+https://github.com/apple/ml-depth-pro.git
!pip -q install git+https://github.com/cvg/GeoCalib.git
!mkdir -p /content/checkpoints
!wget -q -nc https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -P /content/checkpoints

import os, torch
from google.colab import drive

# Mount drive with checks to avoid mount failures if already mounted
try:
    if not os.path.exists('/content/drive/MyDrive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("Google Drive already mounted.")
except Exception as e:
    print(f"[WARNING] Google Drive mount failed: {e}")
    print("Please ensure you grant the required permissions when prompted.")

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
ROOT_ = '/content/drive/MyDrive/cv5570_risk'
need = ['weights/det_ppe_worker_best.pt', 'weights/class_thresholds_ppe_worker.json', 'weights/calib_ppe_worker.joblib', 'code/risk_pose.py']

if os.path.exists(ROOT_):
    for f in need: print(('OK      ' if os.path.exists(f'{ROOT_}/{f}') else 'MISSING '), f)
    print('risk_config.json:', 'found' if os.path.exists(f'{ROOT_}/weights/risk_config.json') else 'MISSING (untuned defaults will be used)')
else:
    print(f"[ERROR] Root directory not found at {ROOT_}. Please check your Drive connection.")

## 2. Engine code (copied from NB4; only change: images are processed in /content/live instead of a Drive folder)

In [ ]:
import os, sys, glob, json, math, importlib
import numpy as np, pandas as pd, cv2, joblib
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT   = '/content/drive/MyDrive/cv5570_risk'
TARGET = 'demo'                      # 'demo' or 'accuracy' (the folder that holds images/ depth/ camera/ [pose/])
W_DIR  = f'{ROOT}/weights'
DET_WEIGHTS = f'{W_DIR}/det_ppe_worker_best.pt'
THR_JSON    = f'{W_DIR}/class_thresholds_ppe_worker.json'
CALIB_PKL   = f'{W_DIR}/calib_ppe_worker.joblib'

BASE = '/content/live'      # demo: uploads are processed in a scratch folder, not on Drive
IMG_DIR, DEPTH_DIR, CAM_DIR, POSE_DIR = f'{BASE}/images', f'{BASE}/depth', f'{BASE}/camera', f'{BASE}/pose'
OUT_DIR = f'{BASE}/results'
for d in (IMG_DIR, OUT_DIR): os.makedirs(d, exist_ok=True)

# shared pose module (written to Drive by the pose notebook)
assert os.path.exists(f'{ROOT}/code/risk_pose.py'), 'Run the pose notebook once first: it writes code/risk_pose.py to Drive.'
sys.path.insert(0, f'{ROOT}/code')
import risk_pose as rp; importlib.reload(rp)

# measured settings from NB7 (safe defaults if the file is missing)
USE_WORLD = None                 # None = follow risk_config.json; True / False forces the YOLO-World second machine detector on / off
USE_QUANTILE_BANDS = True        # use the band edges saved in risk_config.json (if any) instead of 25/50/75
DEFAULTS = dict(det_imgsz=1280, machine_thr=0.2, world=dict(use=False, thr=0.2, model='yolov8s-worldv2.pt',
                prompts=['concrete pump truck', 'crane', 'excavator', 'dump truck', 'loader', 'bulldozer', 'truck', 'construction machine']),
                person_thr=None, head_bins=[0, 10, 16, 24, 36, 1e9], prior_helmet=[0.5] * 5, prior_vest=[0.5] * 5, unverifiable_below_px=0.0, band_edges=None)
CONFIG_PATH = f'{W_DIR}/risk_config.json'
RC = dict(DEFAULTS)
if os.path.exists(CONFIG_PATH): RC.update(json.load(open(CONFIG_PATH))); print('loaded', CONFIG_PATH, '(tuned', RC.get('created'), ')')
else: print('WARNING: risk_config.json not found - using untuned defaults. Run NB7 (diagnostics) first for measured settings.')
if USE_WORLD is not None: RC['world']['use'] = USE_WORLD

CFG = dict(
    # aggregator weights (BLS fatality shares, see the text above)
    W_FALL=0.46, W_PPE=0.29, W_STRUCK=0.25, GAMMA=0.15, FLOOR=0.7,
    # PPE engine
    W_HELMET=0.62, W_VEST=0.19, HELMET_PROX=0.5,          # priors by head size come from risk_config.json
    # struck-by engine
    WORKER_H=1.7, MIN_RADIUS=3.0, MIN_MACHINE_AREA=0.005,
    # fall engine (weights are sensitivity parameters; the posture thresholds live in risk_pose.py)
    W_HEIGHT=0.6, W_POSTURE=0.8,
    # bands: placeholders
    BANDS=[(25, 'Low'), (50, 'Medium'), (75, 'High'), (101, 'Critical')],
    AGREE_TOL=25.0,                     # the two distance methods "agree" if within this % of their mean
    DET_IMGSZ=int(RC['det_imgsz']), MACHINE_THR=float(RC['machine_thr']),
)
if USE_QUANTILE_BANDS and RC.get('band_edges'):
    e = RC['band_edges']; CFG['BANDS'] = [(e[0], 'Low'), (e[1], 'Medium'), (e[2], 'High'), (101, 'Critical')]; print('bands from quantiles:', e)

# detector class -> role (exact names of the fine-tuned model)
ROLE = {'Person': 'person', 'Hardhat': 'helmet', 'NO-Hardhat': 'no_helmet', 'Safety Vest': 'vest',
        'NO-Safety Vest': 'no_vest', 'machinery': 'machine', 'vehicle': 'machine'}      # Mask, NO-Mask, Safety Cone: ignored

In [ ]:
# LOAD MODELS
det   = YOLO(DET_WEIGHTS)
THR   = json.load(open(THR_JSON))
CALIB = joblib.load(CALIB_PKL)
ZONES   = json.load(open(f'{BASE}/zones.json')) if os.path.exists(f'{BASE}/zones.json') else {}
H_KNOWN = json.load(open(f'{BASE}/camera_heights.json')) if os.path.exists(f'{BASE}/camera_heights.json') else {}

_pose = None
def pose_model():
    """the pose model is only loaded if some image has no cached pose/<name>.json"""
    global _pose
    if _pose is None: _pose = rp.load_pose_model()
    return _pose

names = list(det.names.values())
print('detector classes:', names)
missing = [c for c in ROLE if c not in names]
assert not missing, f'these classes are not in the model: {missing} - check ROLE'
print('pose settings:', {k: rp.CFG[k] for k in ('POSE_MODEL', 'POSE_IMGSZ', 'MAX_RANGE_M', 'ELEV_MIN_M')}, '| known camera heights:', H_KNOWN or 'none')

In [ ]:
def calibrated(name, c):
    return float(CALIB[name].predict([c])[0]) if name in CALIB else c

PPE_ROLES = ('helmet', 'no_helmet', 'vest', 'no_vest')
_world = None
def world_model():
    global _world
    if _world is None:
        from ultralytics import YOLOWorld
        _world = YOLOWorld(RC['world']['model']); _world.set_classes(RC['world']['prompts'])
    return _world

def nms_boxes(items, thr=0.5):
    keep = []
    for d in sorted(items, key=lambda d: -d['raw']):
        if all(rp.iou(d['box'], k['box']) < thr for k in keep): keep.append(d)
    return keep

def detect(img):
    """fine-tuned detector (+ optional YOLO-World for machines). PPE: calibrated p used directly (no hard threshold).
    Person: class threshold. Machines: machinery + vehicle merged, one measured threshold."""
    r = det.predict(img, imgsz=CFG['DET_IMGSZ'], conf=0.05, verbose=False)[0]
    out, mach = [], []
    for b in r.boxes:
        name = det.names[int(b.cls)]; role = ROLE.get(name); c = float(b.conf)
        if role is None: continue
        d = dict(role=role, name=name, raw=c, p=calibrated(name, c), box=[float(v) for v in b.xyxy[0]])
        if role == 'machine':
            if c >= CFG['MACHINE_THR']: mach.append(d)
        elif role in PPE_ROLES: out.append(d)
        elif c >= ((RC.get('person_thr') or THR.get(name, 0.25)) if role == 'person' else THR.get(name, 0.25)): out.append(d)
    mach = nms_boxes(mach)
    if RC['world']['use']:
        try:
            wr = world_model().predict(img, imgsz=960, conf=RC['world']['thr'], verbose=False)[0]
            for b in wr.boxes:
                w = dict(role='machine', name='world:' + RC['world']['prompts'][int(b.cls)], raw=float(b.conf), p=float(b.conf), box=[float(v) for v in b.xyxy[0]])
                if all(rp.iou(w['box'], m['box']) < 0.5 for m in mach): mach.append(w)
        except Exception as e:
            print('YOLO-World unavailable, using the fine-tuned detector only:', e); RC['world']['use'] = False
    return out + mach

def get_workers(img, stem, dets):
    """one worker list: full-frame pose people (joints cached by the pose notebook if available, else computed here)
    plus detector people the pose model missed"""
    H, W = img.shape[:2]; persons, source = None, 'inline'
    jp = f'{POSE_DIR}/{stem}.json'
    if os.path.exists(jp):
        d = json.load(open(jp))
        if (d['width'], d['height']) == (W, H) and d.get('pose_model') == rp.CFG['POSE_MODEL'] and d.get('imgsz') == rp.CFG['POSE_IMGSZ']:
            persons, source = rp.raw_persons_from_json(d), 'cached'
    if persons is None:
        persons = rp.run_pose(pose_model(), img)
    det_p = [d for d in dets if d['role'] == 'person']
    return rp.merge_detector_persons(persons, [d['box'] for d in det_p], [d['raw'] for d in det_p]), source

def associate(persons, ppe_dets):
    """each PPE box goes to exactly one worker: smallest worker box containing its centre, in a plausible body zone"""
    for p in persons:
        p['ppe'] = dict(helmet=0.0, no_helmet=0.0, vest=0.0, no_vest=0.0)
    for d in ppe_dets:
        cx, cy = (d['box'][0] + d['box'][2]) / 2, (d['box'][1] + d['box'][3]) / 2
        zone = (0.0, 0.40) if d['role'] in ('helmet', 'no_helmet') else (0.10, 0.75)
        cands = []
        for i, p in enumerate(persons):
            x1, y1, x2, y2 = p['box']; h = max(1.0, y2 - y1)
            if x1 <= cx <= x2 and y1 <= cy <= y2 and zone[0] <= (cy - y1) / h <= zone[1]:
                cands.append(((x2 - x1) * h, i))
        if cands:
            i = min(cands)[1]
            persons[i]['ppe'][d['role']] = max(persons[i]['ppe'][d['role']], d['p'])

def filter_machines(dets, shape):
    H, W = shape[:2]
    return [d for d in dets if d['role'] == 'machine'
            and (d['box'][2] - d['box'][0]) * (d['box'][3] - d['box'][1]) >= CFG['MIN_MACHINE_AREA'] * H * W]

feet_px = rp.feet_px

In [ ]:
_cam_cache, _depth_cache = {}, {}
def load_camera(stem):
    if stem not in _cam_cache:
        p = f'{CAM_DIR}/{stem}.json'
        _cam_cache[stem] = json.load(open(p)) if os.path.exists(p) else None
    return _cam_cache[stem]

def load_depth(stem):
    if stem not in _depth_cache:
        p = f'{DEPTH_DIR}/{stem}.npz'
        if os.path.exists(p):
            z = np.load(p); _depth_cache[stem] = (z['depth'].astype(np.float32), float(z['f_px']))
        else:
            _depth_cache[stem] = None
    return _depth_cache[stem]

def build_ctx(an, method):
    """decides which distance mode is actually usable for this image; 'pixel' is the rough fallback"""
    img, stem, cam, scene = an['img'], an['stem'], an['cam'], an['scene']
    H, W = img.shape[:2]
    ctx = dict(stem=stem, H=H, W=W, method=method, mode='pixel', tier='low', h=None, spread=None, n_ref=0,
               cam=None, depth=None, f_px=None, f_geocalib=None, f_depthpro=None, note=an['cam_note'])
    raw_cam, dep = load_camera(stem), load_depth(stem)
    if raw_cam is not None: ctx['f_geocalib'] = raw_cam['fx']
    if dep is not None:
        ctx['f_depthpro'] = dep[1]
        if dep[0].shape != (H, W): ctx['note'] = 'Depth Pro map size differs from image'; dep = None
    if method == 'geometry':
        if cam is None: ctx['note'] = ctx['note'] or 'no GeoCalib output for this image'
        elif rp.is_topdown(cam): ctx['note'] = 'camera looks almost straight down'
        elif scene['h_ref'] is None: ctx['note'] = scene['status'] or 'no upright full-body worker to set the scale'
        else:
            sp, n = scene['spread'], scene['n_ref']
            known = scene['ref_source'] == 'known camera height'
            ctx.update(mode='geometry', cam=cam, h=scene['h_ref'], spread=sp, n_ref=n,
                       tier='high' if (known or (n >= 3 and sp is not None and sp <= 0.15)) else 'medium')
    elif method == 'depthpro':
        if dep is None: ctx['note'] = ctx['note'] or 'no Depth Pro output for this image'
        else: ctx.update(mode='depthpro', depth=dep[0], f_px=dep[1], tier='medium')
    return ctx

def depth_xyz(u, v, ctx):
    D, f = ctx['depth'], ctx['f_px']; Hd, Wd = D.shape
    ui, vi = int(np.clip(round(u), 0, Wd - 1)), int(np.clip(round(v), 0, Hd - 1))
    Z = float(np.median(D[max(0, vi - 2):vi + 3, max(0, ui - 2):ui + 3]))
    return np.array([(ui - Wd / 2) * Z / f, (vi - Hd / 2) * Z / f, Z])

def machine_pt(u, v, ctx):
    """a machine's base point (machines are assumed to stand on the ground plane)"""
    return rp.floor_point(u, v, ctx['cam'], ctx['h']) if ctx['mode'] == 'geometry' else depth_xyz(u, v, ctx)

def feet3d(wk, ctx):
    fu, fv = feet_px(wk)
    if ctx['mode'] == 'geometry':
        e = wk['elev']
        if e['kind'] == 'above_camera': return None          # feet at/above eye level: no plane to place them on
        h = e['h_i'] if (e['kind'] == 'elevated' and e['h_i'] is not None) else ctx['h']      # elevated: their OWN plane
        return rp.floor_point(fu, fv, ctx['cam'], h)
    return depth_xyz(fu, fv, ctx)

def worker_machine_dist(wk, m, ctx):
    """-> (distance_m, machine_width_m, mode used)"""
    x1, y1, x2, y2 = m['box']
    if ctx['mode'] in ('geometry', 'depthpro'):
        pts = [feet3d(wk, ctx), machine_pt((x1 + x2) / 2, y2, ctx), machine_pt(x1, y2, ctx), machine_pt(x2, y2, ctx)]
        if all(p is not None for p in pts):
            return float(np.linalg.norm(pts[0] - pts[1])), float(np.linalg.norm(pts[2] - pts[3])), ctx['mode']
    fu, fv = feet_px(wk)
    k = CFG['WORKER_H'] / max(1.0, wk['box'][3] - wk['box'][1])        # pixel fallback: worker box height = 1.7 m
    return math.hypot(fu - (x1 + x2) / 2, fv - y2) * k, (x2 - x1) * k, 'pixel'

In [ ]:
def struck_engine(wk, machines, ctx):
    if not machines:
        return dict(score=0.0, tier='no_machines_found', n_machines=0, nearest_m=None, nearest_box=None, mode=ctx['mode'])
    probs, best = [], None
    for m in machines:
        d, w, mode = worker_machine_dist(wk, m, ctx)
        probs.append(math.exp(-(d / max(CFG['MIN_RADIUS'], w)) ** 2))
        if best is None or d < best[0]: best = (d, mode, m['box'])
    score = 100 * (1 - np.prod([1 - s for s in probs]))
    tier = ctx['tier'] if best[1] == ctx['mode'] else 'low'
    return dict(score=float(score), tier=tier, n_machines=len(machines), nearest_m=float(best[0]), nearest_box=best[2], mode=best[1])

def head_prior(head_eff):
    """measured P(PPE really missing | no box found) for this head size (head height in px at detector resolution)"""
    edges = RC['head_bins']; i = next((k for k in range(len(edges) - 1) if edges[k] <= head_eff < edges[k + 1]), len(edges) - 2)
    return float(RC['prior_helmet'][i]), float(RC['prior_vest'][i])

def ppe_engine(wk, prox):
    e = wk['ppe']; evidence = any(e[k] > 0 for k in ('helmet', 'no_helmet', 'vest', 'no_vest'))
    if not evidence and wk['head_eff'] < RC['unverifiable_below_px']:
        return dict(score=None, m_helmet=None, m_vest=None, missing=[], status='unverifiable')
    pr_h, pr_v = head_prior(wk['head_eff'])
    m_h = (1 - e['helmet']) * (e['no_helmet'] + (1 - e['no_helmet']) * pr_h)
    m_v = (1 - e['vest']) * (e['no_vest'] + (1 - e['no_vest']) * pr_v)
    score = 100 * (CFG['W_HELMET'] * (1 + CFG['HELMET_PROX'] * prox) * m_h + CFG['W_VEST'] * (1 + prox) * m_v)
    return dict(score=float(min(100, score)), m_helmet=m_h, m_vest=m_v, status='evidence' if evidence else 'prior',
                missing=[n for n, m in (('helmet', m_h), ('vest', m_v)) if m > 0.5])

def fall_engine(wk, zones):
    """height above the ground (estimated elevation, optionally hand-drawn zones) x unstable posture"""
    ps, e = wk['posture'], wk['elev']
    sev_z, in_zone = 0.0, False
    fu, fv = feet_px(wk)
    for z in zones:
        if cv2.pointPolygonTest(np.array(z['polygon'], dtype=np.int32), (float(fu), float(fv)), False) >= 0:
            in_zone = True; sev_z = max(sev_z, min(1.0, z['drop_m'] / rp.CFG['ELEV_FULL_M']))
    sev = max(e['sev'] or 0.0, sev_z)
    score = 100 * (1 - (1 - CFG['W_HEIGHT'] * sev) * (1 - CFG['W_POSTURE'] * (ps['value'] or 0.0)))
    return dict(score=float(score), posture=ps['label'], in_zone=in_zone, severity=sev)

def band(r):
    return next(n for t, n in CFG['BANDS'] if r < t)

def aggregate(fall, ppe, struck, evidence):
    """evidence = (fall_has_evidence, ppe_has_evidence, struck_has_evidence): only engines with real evidence set the floor"""
    a = np.array([fall, ppe or 0.0, struck]) / 100
    base = CFG['W_FALL'] * a[0] + CFG['W_PPE'] * a[1] + CFG['W_STRUCK'] * a[2]
    pair = a[0] * a[1] + a[0] * a[2] + a[1] * a[2]
    ev_max = max([x for x, ok in zip(a, evidence) if ok] or [0.0])
    R = float(min(1.0, max(base + CFG['GAMMA'] * pair, CFG['FLOOR'] * ev_max)) * 100)
    return dict(risk=R, band=band(R))

def elevation_text(e):
    if e['kind'] == 'elevated': return f"elevated {e['m']:.1f} m"
    if e['kind'] == 'above_camera': return f"above camera level (>= {e['m']:.1f} m)"
    if e['kind'] == 'ground': return 'ground'
    if e['kind'] == 'below_ground': return f"below ground {-e['m']:.1f} m"
    return 'unknown'

In [ ]:
def analyse_image(path):
    """the slow part (detector + pose + association + elevation) - done once per image, reused for both distance methods"""
    img = cv2.imread(path); stem = os.path.splitext(os.path.basename(path))[0]
    dets = detect(img)
    persons, pose_source = get_workers(img, stem, dets)
    cam, cam_note = load_camera(stem), ''
    if cam is not None and (cam['width'], cam['height']) != (img.shape[1], img.shape[0]):
        cam, cam_note = None, 'GeoCalib size differs from image'
    scene = rp.analyse_persons(persons, cam, img.shape, H_KNOWN.get(stem))
    for p_ in persons: p_['head_eff'] = rp.head_px(p_) * CFG['DET_IMGSZ'] / max(img.shape[:2])        # head height in px at detector resolution
    associate(persons, [d for d in dets if d['role'] in ('helmet', 'no_helmet', 'vest', 'no_vest')])
    return dict(stem=stem, img=img, persons=persons, machines=filter_machines(dets, img.shape), cam=cam, scene=scene,
                pose_source=pose_source, cam_note=cam_note)

def score_image(an, method):
    ctx = build_ctx(an, method)
    zones = ZONES.get(an['stem'], [])
    rows, extras = [], []
    for i, w in enumerate(an['persons']):
        sb = struck_engine(w, an['machines'], ctx)
        pp = ppe_engine(w, sb['score'] / 100)
        fl = fall_engine(w, zones)
        e = w['elev']
        ev = (e['kind'] != 'unknown' or fl['in_zone'], pp['status'] == 'evidence', sb['n_machines'] > 0)
        ag = aggregate(fl['score'], pp['score'], sb['score'], ev)
        n_ev = int(ev[1]) + int(ev[2]) + int(e['kind'] != 'unknown')
        rows.append(dict(image=an['stem'], worker=i, risk=round(ag['risk'], 1), band=ag['band'], risk_confidence=('high', 'medium', 'low')[0 if n_ev >= 2 else (1 if n_ev == 1 else 2)],
                         fall=round(fl['score'], 1), ppe=None if pp['score'] is None else round(pp['score'], 1), ppe_status=pp['status'], struck=round(sb['score'], 1),
                         missing=','.join(pp['missing']), posture=fl['posture'], elevation=elevation_text(e),
                         elev_m=None if e['m'] is None else round(e['m'], 2), elev_confidence=None if e['kind'] == 'unknown' else e['conf'],
                         nearest_m=None if sb['nearest_m'] is None else round(sb['nearest_m'], 2),
                         dist_mode=sb['mode'], dist_confidence=sb['tier'], n_machines=sb['n_machines'], worker_source=w['src'],
                         head_px_eff=round(w['head_eff'], 1)))
        extras.append(dict(feet=feet_px(w), nearest_box=sb['nearest_box']))
    return rows, extras, ctx

def agreement(rows_g, ctx_g, rows_d, ctx_d):
    """label-free check: do the two methods give the same nearest worker-machine distance?  -> % difference per worker"""
    out = []
    for rg, rd in zip(rows_g, rows_d):
        a, b = rg['nearest_m'], rd['nearest_m']
        ok = a is not None and b is not None and rg['dist_mode'] == 'geometry' and rd['dist_mode'] == 'depthpro' and (a + b) > 0
        out.append(round(100 * abs(a - b) / ((a + b) / 2), 1) if ok else None)
    return out

BAND_COL = {'Low': (60, 180, 75), 'Medium': (0, 200, 255), 'High': (0, 128, 255), 'Critical': (0, 0, 230)}   # BGR

def draw(an, rows, extras, ctx):
    vis = an['img'].copy(); H, W = vis.shape[:2]
    fs = max(0.5, W / 1600); th = max(1, int(round(W / 800)))
    for m in an['machines']:
        x1, y1, x2, y2 = [int(v) for v in m['box']]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 255), th + 1)
        cv2.putText(vis, m['name'], (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 0, 255), th)
    for w, r, ex in zip(an['persons'], rows, extras):
        x1, y1, x2, y2 = [int(v) for v in w['box']]; col = BAND_COL[r['band']]
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, th + 1)
        cv2.putText(vis, f"W{r['worker']} {r['risk']:.0f} {r['band']}", (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, col, th)
        tag = f"F{r['fall']:.0f} P{'n/a' if r['ppe'] is None else format(r['ppe'], '.0f')} S{r['struck']:.0f}"
        if w['elev']['kind'] in ('elevated', 'above_camera'):
            tag += f" | UP {w['elev']['m']:.1f}m" if w['elev']['kind'] == 'elevated' else f" | UP >={w['elev']['m']:.1f}m"
        cv2.putText(vis, tag, (x1, min(H - 5, y2 + int(22 * fs))), cv2.FONT_HERSHEY_SIMPLEX, fs * 0.8, col, max(1, th - 1))
        if ex['nearest_box'] is not None and r['nearest_m'] is not None:
            bx = ex['nearest_box']; p0 = (int(ex['feet'][0]), int(ex['feet'][1])); p1 = (int((bx[0] + bx[2]) / 2), int(bx[3]))
            cv2.line(vis, p0, p1, col, th)
            cv2.putText(vis, f"{r['nearest_m']:.1f} m", ((p0[0] + p1[0]) // 2, (p0[1] + p1[1]) // 2), cv2.FONT_HERSHEY_SIMPLEX, fs, col, th)
    return vis

## 3. Live Depth Pro + GeoCalib (same code and output format as NB2 / NB3)

In [ ]:
import numpy as np, torch, cv2, json, time, depth_pro
from geocalib import GeoCalib
os.chdir('/content')   # Depth Pro looks for ./checkpoints/depth_pro.pt
dp_model, dp_transform = depth_pro.create_model_and_transforms(device=torch.device('cuda'), precision=torch.half); dp_model.eval()
gc_model = GeoCalib().to('cuda')
for d in (DEPTH_DIR, CAM_DIR): os.makedirs(d, exist_ok=True)

def run_depth_pro(path, stem):
    H, W = cv2.imread(path).shape[:2]
    image, _, f_px = depth_pro.load_rgb(path)
    with torch.no_grad(): pred = dp_model.infer(dp_transform(image), f_px=f_px)
    depth = pred['depth'].squeeze().float().cpu().numpy()
    assert depth.shape == (H, W), f'depth map {depth.shape[::-1]} vs image {(W, H)}'
    np.savez_compressed(f'{DEPTH_DIR}/{stem}.npz', depth=np.clip(depth, 0, 60000).astype(np.float16), f_px=float(pred['focallength_px']))

def run_geocalib(path, stem):
    img = gc_model.load_image(path).to('cuda'); H, W = int(img.shape[-2]), int(img.shape[-1])
    with torch.no_grad(): res = gc_model.calibrate(img)
    cam, grav = res['camera'], res['gravity']
    f = cam.f[0].cpu().numpy(); c = cam.c[0].cpu().numpy()
    g = grav.vec3d[0].cpu().numpy().astype(float); g = g / np.linalg.norm(g)
    roll, pitch = [float(np.degrees(v)) for v in grav.rp[0].cpu().numpy()]
    json.dump(dict(stem=stem, width=W, height=H, fx=float(f[0]), fy=float(f[1]), cx=float(c[0]), cy=float(c[1]), g=g.tolist(),
                   g_convention='up vector, camera frame x right / y down / z forward', roll_deg=roll, pitch_deg=pitch,
                   near_topdown=bool(-g[2] > 0.85)), open(f'{CAM_DIR}/{stem}.json', 'w'))
print('Depth Pro + GeoCalib loaded')

## 4. Pipeline for one uploaded image

In [ ]:
import pandas as pd
MAX_SIDE = 1920          # larger uploads are shrunk (speed); the same file feeds every model so sizes always match
LIVE = {}                # stem -> dict(an, out, agr)

def process_upload(rgb, cam_h=None):
    """RGB numpy image -> runs Depth Pro, GeoCalib, detector, pose, engines (both distance methods). Returns the stem."""
    stem = f'up{int(time.time() * 1000)}'
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR); h, w = bgr.shape[:2]
    if max(h, w) > MAX_SIDE:
        k = MAX_SIDE / max(h, w); bgr = cv2.resize(bgr, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA)
    path = f'{IMG_DIR}/{stem}.jpg'; cv2.imwrite(path, bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if cam_h: H_KNOWN[stem] = float(cam_h)
    t = {}
    t0 = time.time(); run_depth_pro(path, stem); t['Depth Pro'] = time.time() - t0
    t0 = time.time(); run_geocalib(path, stem);  t['GeoCalib'] = time.time() - t0
    t0 = time.time(); an = analyse_image(path); t['detector + pose'] = time.time() - t0
    out = {m: score_image(an, m) for m in ('geometry', 'depthpro')}
    agr = agreement(out['geometry'][0], out['geometry'][2], out['depthpro'][0], out['depthpro'][2])
    for m in out:
        for r, a in zip(out[m][0], agr): r['agree_pct'] = a
    LIVE[stem] = dict(an=an, out=out, t=t)
    return stem

COLS = ['worker', 'risk', 'band', 'risk_confidence', 'fall', 'ppe', 'struck', 'missing', 'posture', 'elevation', 'nearest_m', 'dist_mode', 'dist_confidence']

def render(stem, method):
    """-> (annotated RGB image, table, markdown notes) for the chosen distance method"""
    if not stem or stem not in LIVE: return None, pd.DataFrame(columns=COLS), 'Upload an image and press **Analyse**.'
    an = LIVE[stem]['an']; rows, extras, ctx = LIVE[stem]['out'][method]
    vis = cv2.cvtColor(draw(an, rows, extras, ctx), cv2.COLOR_BGR2RGB)
    df = pd.DataFrame(rows)[COLS] if rows else pd.DataFrame(columns=COLS)
    sc = an['scene']; n = []
    if rows:
        top = max(rows, key=lambda r: r['risk'])
        n.append(f"**{len(rows)} worker(s)**, {len(an['machines'])} machine(s). Highest risk: **W{top['worker']} = {top['risk']:.0f} ({top['band']})**.")
    else:
        n.append(f"No workers found. {len(an['machines'])} machine(s) detected.")
    if ctx['mode'] != method:
        n.append(f"**Note:** {method} could not be used here ({ctx['note'] or 'unknown reason'}), so distances are rough pixel estimates (low confidence).")
    elif method == 'geometry':
        s_ = 'n/a' if ctx['spread'] is None else f"{ctx['spread']:.0%}"
        n.append(f"Geometry: camera height {ctx['h']:.2f} m from {ctx['n_ref']} ground worker(s) (spread {s_}).")
    f1, f2 = ctx['f_geocalib'], ctx['f_depthpro']
    if f1 and f2:
        n.append(f"Focal length: GeoCalib {f1:.0f}px vs Depth Pro {f2:.0f}px (ratio {f2 / f1:.2f})." + ('' if 0.7 <= f2 / f1 <= 1.4 else ' **The two disagree strongly - treat distances with care.**'))
    ag = [r['agree_pct'] for r in rows if r['agree_pct'] is not None]
    if ag:
        n.append(f"Method agreement on nearest worker-machine distance: median difference {np.median(ag):.0f}% -> " + ('agree' if np.median(ag) <= CFG['AGREE_TOL'] else '**disagree**'))
    n.append(f"Elevation: {sc['status'] if sc['status'] else 'not available'}" + (f" (camera height {sc['h_ref']:.2f} m)" if sc['h_ref'] and sc['elev_ok'] else ''))
    t = LIVE[stem]['t']; n.append('Timing: ' + ', '.join(f'{k} {v:.1f}s' for k, v in t.items()))
    return vis, df, '\n\n'.join('- ' + x for x in n)
print('pipeline ready')

## 5. Warm-up (loads YOLO-World / pose once and checks the whole chain on a sample) then the Gradio app

In [ ]:
# warm-up: run one image so the first live upload is fast; uses a demo image from Drive if present, else a blank frame
import glob
_s = sorted(glob.glob(f'{ROOT}/demo/images/*.jpg') + glob.glob(f'{ROOT}/demo/images/*.png'))
_img = cv2.cvtColor(cv2.imread(_s[0]), cv2.COLOR_BGR2RGB) if _s else np.full((720, 1280, 3), 127, np.uint8)
_st = process_upload(_img); print('warm-up ok:', LIVE[_st]['t'])

In [ ]:
import gradio as gr
#EXAMPLES = sorted(glob.glob(f'{ROOT}/demo/images/*.jpg') + glob.glob(f'{ROOT}/demo/images/*.png'))[:9]

def on_analyse(img, method, cam_h):
    if img is None: return None, None, pd.DataFrame(columns=COLS), 'Please upload an image first.'
    try:
        stem = process_upload(img, cam_h if cam_h and cam_h > 0 else None)
    except Exception as e:
        return None, None, pd.DataFrame(columns=COLS), f'**Error:** {type(e).__name__}: {e}'
    return (stem,) + render(stem, method)

def on_method(stem, method):
    return render(stem, method)

with gr.Blocks(title='Construction site risk scoring', theme=gr.themes.Soft()) as app:
    gr.Markdown('## Construction-site worker risk scoring\nPPE (helmet / vest) + struck-by (distance to machines) + fall (height & posture) -> one risk score per worker. '
                'Distances come from **GeoCalib ground-plane geometry** or **Depth Pro**.')
    state = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.Image(type='numpy', label='Site photo', height=360)
            method = gr.Radio([('GeoCalib (ground-plane geometry)', 'geometry'), ('Depth Pro (metric depth)', 'depthpro')], value='geometry', label='Distance method')
            cam_h = gr.Number(label='Camera height above floor in metres (optional, if known)', value=None)
            btn = gr.Button('Analyse', variant='primary')
        with gr.Column(scale=2):
            outimg = gr.Image(label='Result (box colour = risk band; F/P/S = fall / PPE / struck-by score)', height=520)
            notes = gr.Markdown()
    table = gr.Dataframe(label='Per-worker scores (fall / ppe / struck 0-100; risk = weighted combination)', wrap=True)
    gr.Markdown('Weights: fall 0.46, PPE 0.29, struck-by 0.25 (+0.15 interaction term), from BLS 2023 construction fatality shares. Scores are rule-based; see the report for limits. '
                'Bands are placeholders (25 / 50 / 75).')
    btn.click(on_analyse, [inp, method, cam_h], [state, outimg, table, notes])
    method.change(on_method, [state, method], [outimg, table, notes])

app.launch(share=True, debug=True)